In [1]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Heat Pumps and Thermal Storage

In this example, a heat demand is supplied by a wind turbine in combination with a heat pump and a water tank that stores hot water with a standing loss.

In [2]:
import pandas as pd

import pypsa

n = pypsa.Network()
n.set_snapshots(pd.date_range("2025-01-01 00:00", "2025-01-01 03:00", freq="h"))

n.add("Bus", "power", carrier="AC")
n.add("Bus", "heat", carrier="heat")

n.add(
    "Generator",
    "wind turbine",
    bus="power",
    carrier="wind",
    p_nom_extendable=True,
    p_max_pu=[0.0, 0.2, 0.7, 0.4],
    capital_cost=500,
)

n.add("Load", "heat demand", bus="heat", p_set=20);

/tmp/ipykernel_7980/1312632822.py:5: PyPSA2Warning: pandas reads strings as `str` dtype since 3.0. PyPSA converts them back to object dtype on import, but keeps `str` from PyPSA 2.0 on. Set `pypsa.options.future.string_dtype` to True to opt in now or to False to keep the current behaviour.
  n = pypsa.Network()


The heat pump has time-varying efficiency (i.e. its coefficient of performance, COP)  due to changing ambient temperatures.

In [3]:
n.add(
    "Link",
    "heat pump",
    bus0="power",
    bus1="heat",
    efficiency=[2.5, 3.0, 3.2, 3.0],
    capital_cost=1000,
    p_nom_extendable=True,
);

The hot water tank has a standing loss of 1% of its state of charge per hour.

In [4]:
n.add(
    "Store",
    "water tank",
    bus="heat",
    e_cyclic=True,
    e_nom=100,
    standing_loss=0.01,
);

The wind turbine and the heat pump can be sized by the optimisation, while the water tank has a fixed size of 100 MWh.

In [5]:
n.optimize();

/tmp/ipykernel_7980/1450685117.py:1: PyPSA2Warning: From PyPSA 2.0 on, the objective leaves out the capital costs of existing capacity on extendable assets, which improves LP conditioning. Pass `include_objective_constant` to `n.optimize` to choose per call. Set `pypsa.options.future.exclude_objective_constant` to True to opt in now or to False to keep the current behaviour.
  n.optimize();
Index(['power', 'heat'], dtype='object', name='name')


Index(['wind turbine'], dtype='object', name='name')


Index(['heat pump'], dtype='object', name='name')


Index(['water tank'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


... (7 outputs left out)

obj: 46 rows, 22 columns, 69 nonzeros


Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 22 primals, 54 duals
Objective: 2.35e+04
Solver: gurobi
Runtime: 0.00s
Dual bound: 2.35e+04
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-fix-p-lower, Store-fix-p-upper, Store-fix-p_store-lower, Store-fix-p_store-upper, Store-energy_balance were not assigned to the network.


In [6]:
pd.DataFrame({attr: n.stores_t[attr]["water tank"] for attr in ["p", "e"]}).round(3)

,p,e
snapshot,,
2025-01-01 00:00:00,20.000,4.377
2025-01-01 01:00:00,4.333,0.000
2025-01-01 02:00:00,-13.423,13.423
2025-01-01 03:00:00,-11.334,24.623


In [7]:
pd.DataFrame({attr: n.links_t[attr]["heat pump"] for attr in ["p0", "p1"]}).round(3)

,p0,p1
snapshot,,
2025-01-01 00:00:00,0.000,-0.000
2025-01-01 01:00:00,5.222,-15.667
2025-01-01 02:00:00,10.445,-33.423
2025-01-01 03:00:00,10.445,-31.334
